# 10b · Supplement on val: training targets and distillation

Section 10 of `10_report` on its own, for when the main tables (sections 1–9) need no rerun. It reads **val scores only**: no test score is read, so the supplement runs stay blind on test. The code of the comparison cell is the same as in `10_report`.

**Kaggle settings:** CPU, no internet, a few minutes. Also runs locally from `notebooks/` (inputs read from `../../kaggle_outputs`).

**Inputs** (Add Input):
- Dataset `wm-src`, a version with `scores/` (val scores of `d1_minilm_main_mse_s42/43`, `d1_deberta_main_mse_s42/43`, `d2_deberta_main_ce4_s42` and `d3_ce_debertalarge_main_ce4`, the teacher). Required.
- Notebook Output **01_preprocess** (`pairs.parquet`). Required.
- Notebook Outputs **14_distill** and **15_deberta_base_s43**. A comparison whose scores are missing is skipped with a message.

**Output** (`/kaggle/working/report/`, locally `data/report/`): `supplement_val.csv`.

In [ ]:
# Setup: make src/ importable (dataset wm-src, any version: only src/data.py and src/evaluation.py are used).
import glob, sys
from pathlib import Path

_hits = sorted(h for h in glob.glob('/kaggle/input/**/evaluation.py', recursive=True)
               if Path(h).with_name('data.py').exists()) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))
print('src from', ROOT)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

from src.data import PREPROCESS_HINT, find
from src.evaluation import paired_bootstrap, summarize

ON_KAGGLE = Path('/kaggle/input').exists()
INPUT = '/kaggle/input' if ON_KAGGLE else '../../kaggle_outputs'
OUT = Path('/kaggle/working/report') if ON_KAGGLE else ROOT / 'data' / 'report'
SCORES = {Path(p).name: p for p in glob.glob(f'{INPUT}/**/scores/*_val.parquet', recursive=True)}   # <run>_val.parquet
CE5L = 'd3_ce_debertalarge_main_ce4'    # teacher of notebook 14: DeBERTa-large CE, mean of 5 fold models

pairs = pd.read_parquet(find('pairs.parquet', PREPROCESS_HINT, INPUT),
                        columns=['example_id', 'query_id', 'esci_label', 'split'])
parts = {'val': pairs[pairs['split'] == 'val'].sort_values(['query_id', 'example_id']).reset_index(drop=True)}


def scores(run, split):
    sc = pd.read_parquet(SCORES[f'{run}_{split}.parquet'], columns=['example_id', 'score'])
    s = parts[split][['example_id']].merge(sc, on='example_id', how='left')['score']
    assert s.notna().all(), f'{run} {split}: scores do not cover every pair'
    return s.to_numpy()


metrics, per_query = {}, {}
print(f"{len(SCORES)} val score files | {parts['val']['query_id'].nunique():,} val queries")

## Supplement: training targets and distillation (val only)

Two questions, each pooled over seeds 42 and 43:

- **ce4 vs MSE on DeBERTa-base main.** D2 compared them on seed 42 only; seed 43 of ce4 comes from notebook 15 (the MSE runs from notebook 05).
- **Distillation vs the MiniLM baselines.** Notebook 14 compares each distilled MiniLM (`kd05`: alpha 0.5, `kd00`: teacher probabilities only) with a ce4 MiniLM trained in the same notebook. For MiniLM, ce4 is a weaker baseline than MSE (D2), so the same students are also compared with **MiniLM main MSE** (`d1_minilm_main_mse`, notebook 05): same model, text, length, learning rate, warm-up and batches; only the training target differs.

How to read the table:
- **Val only.** The supplement runs are ablations of the system, not candidates for it, so their test scores stay blind.
- **Pooled over seeds:** per query, the mean over seeds of (b − a); 95% bootstrap CI over queries (the same computation as in notebook 14, so `kd00 vs ce4` must match `d4_pooled.csv`). The CI covers the sampling of queries, not the training randomness: `per_seed` and `same_sign` show whether both seeds agree, and the `seed noise` rows give the s43 − s42 difference of one configuration for comparison (DeBERTa-base ce4 over five seeds: `deberta_base_seeds.csv` of notebook 15).
- **Recovered share:** (student − MSE) / (teacher − MSE) on the per-query nDCG averaged over seeds, with the teacher = DeBERTa-large CE, mean of 5 fold models (the out-of-fold teacher of notebook 14).

In [ ]:
SUPP_SEEDS = [42, 43]
SUPP = [   # (question, run a, run b), {s} = seed: paired difference b - a on val
    ('D2 DeBERTa-base main: ce4 vs MSE', 'd1_deberta_main_mse_s{s}', 'd2_deberta_main_ce4_s{s}'),
    ('D4 MiniLM main: ce4 vs MSE', 'd1_minilm_main_mse_s{s}', 'd4_minilm_main_ce4_s{s}'),
    ('D4 MiniLM main: distilled (alpha 0.5) vs MSE', 'd1_minilm_main_mse_s{s}', 'd4_minilm_main_kd05_s{s}'),
    ('D4 MiniLM main: distilled (alpha 0) vs MSE', 'd1_minilm_main_mse_s{s}', 'd4_minilm_main_kd00_s{s}'),
    ('D4 MiniLM main: distilled (alpha 0) vs ce4 (as in notebook 14)', 'd4_minilm_main_ce4_s{s}', 'd4_minilm_main_kd00_s{s}'),
]
NOISE = ['d1_deberta_main_mse_s{s}', 'd2_deberta_main_ce4_s{s}',
         'd1_minilm_main_mse_s{s}', 'd4_minilm_main_ce4_s{s}', 'd4_minilm_main_kd00_s{s}']   # s43 - s42


def available(*runs):
    """True if every run has val scores in the inputs (each {s} filled with every seed)."""
    return all(f'{r.format(s=s)}_val.parquet' in SCORES for r in runs for s in SUPP_SEEDS)


def val_ndcg(run):
    """Per-query val nDCG of a run, indexed by query_id (scored once and cached)."""
    if (run, 'val') not in per_query:
        metrics[run, 'val'], per_query[run, 'val'] = summarize(parts['val'], scores(run, 'val'))
    return per_query[run, 'val'].set_index('query_id')['ndcg']


def boot_ci(x, n_boot=1000, seed=42):
    m = x[np.random.default_rng(seed).integers(0, len(x), (n_boot, len(x)))].mean(axis=1)
    return [round(float(np.percentile(m, 2.5)), 4), round(float(np.percentile(m, 97.5)), 4)]


def pooled(a, b):
    """b - a pooled over SUPP_SEEDS: per query, the mean over seeds of (b_s - a_s)."""
    d = pd.concat({s: val_ndcg(b.format(s=s)) - val_ndcg(a.format(s=s)) for s in SUPP_SEEDS}, axis=1, join='inner').dropna()
    x = d.mean(axis=1).to_numpy()
    per_seed = [round(float(d[s].mean()), 4) for s in SUPP_SEEDS]
    return {'diff': round(float(x.mean()), 4), 'ci95': boot_ci(x), 'per_seed': per_seed,
            'same_sign': len({np.sign(v) for v in per_seed}) == 1,
            'wilcoxon_p': f'{wilcoxon(x[x != 0]).pvalue:.1e}', 'queries': len(x)}


def recovered(student, base='d1_minilm_main_mse_s{s}', n_boot=1000, seed=42):
    """(student - base) / (teacher - base) on per-query val nDCG averaged over seeds, with a bootstrap CI."""
    def over_seeds(r):
        return pd.concat([val_ndcg(r.format(s=s)) for s in SUPP_SEEDS], axis=1, join='inner').mean(axis=1)
    d = pd.concat({'s': over_seeds(student), 'r': over_seeds(base), 't': val_ndcg(CE5L)},
                  axis=1, join='inner').dropna().to_numpy()
    share = lambda m: (m[..., 0] - m[..., 1]) / (m[..., 2] - m[..., 1])
    boot = share(d[np.random.default_rng(seed).integers(0, len(d), (n_boot, len(d)))].mean(axis=1))
    return round(float(share(d.mean(axis=0))), 3), [round(float(np.percentile(boot, q)), 3) for q in (2.5, 97.5)]


rows, used = [], set()
for q, a, b in SUPP:
    if available(a, b):
        rows.append({'question': q, 'a': a.format(s='*'), 'b': b.format(s='*'), **pooled(a, b)})
        used |= {r.format(s=s) for r in (a, b) for s in SUPP_SEEDS}
    else:
        print(f'skipped (val scores not in the inputs): {q}')
for run in NOISE:
    if available(run):
        val_ndcg(run.format(s=42)), val_ndcg(run.format(s=43))   # scores them if not done yet
        p = paired_bootstrap(per_query[run.format(s=42), 'val'], per_query[run.format(s=43), 'val'])
        rows.append({'question': 'seed noise: s43 vs s42', 'a': run.format(s=42), 'b': run.format(s=43),
                     'diff': p['diff'], 'ci95': p['ci95']})
supplement = pd.DataFrame(rows) if rows else None
if supplement is not None:
    display(supplement)
if used:
    display(pd.DataFrame([{'run': r, 'val_ndcg': metrics[r, 'val']['ndcg'], 'val_ci95': metrics[r, 'val']['ndcg_ci95']}
                          for r in sorted(used)]))
for v in ['kd05', 'kd00']:
    if available(f'd4_minilm_main_{v}_s{{s}}', 'd1_minilm_main_mse_s{s}', CE5L):
        share, ci = recovered(f'd4_minilm_main_{v}_s{{s}}')
        print(f'{v}: recovers {share:.1%} of the val gap between MiniLM main MSE and the teacher '
              f'(95% CI {ci[0]:.1%} to {ci[1]:.1%})')

## Save

In [ ]:
if supplement is not None:
    OUT.mkdir(parents=True, exist_ok=True)
    supplement.to_csv(OUT / 'supplement_val.csv', index=False)
    print('saved', OUT / 'supplement_val.csv')